
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# Demo - Setting Up and Managing Serverless Lakeflow Jobs  

In this demo, we’ll set up a **Databricks Lakeflow Jobs** leveraging SQL tasks to automate a series of **Data Warehousing** tasks. These tasks include data ingestion, validation, transformation, and generating insights within a **Medallion Architecture** (Bronze, Silver, Gold layers). Additionally, we will explore error handling, retries, scheduling options, and integration with external tools.

**Learning Objectives**

By the end of this demo, you will learn how to:

- Create a Lakeflow Job with **SQL tasks** for a Medallion Architecture pipeline.
- Set up task dependencies and implement conditional logic for lakeflow jobs control.
- Use the Lakeflow Job UI for **monitoring** and **data lineage visualization** to trace data transformations and dependencies.
- Configure error handling and retries for tasks.
- Schedule Lakeflow Job using manual triggers.
- Set up **notifications** for monitoring and analyze the execution history.

## REQUIRED - SELECT CLASSIC COMPUTE
Before executing cells in this notebook, please select your classic compute cluster in the lab. Be aware that **Serverless** is enabled by default.

Follow these steps to select the classic compute cluster:
1. Navigate to the top-right of this notebook and click the drop-down menu to select your cluster. By default, the notebook will use **Serverless**.

1. If your cluster is available, select it and continue to the next cell. If the cluster is not shown:
    - In the drop-down, select **More**.
    - In the **Attach to an existing compute resource** pop-up, select the first drop-down. You will see a unique cluster name in that drop-down. Please select that cluster.
    
**NOTE:** If your cluster has terminated, you might need to restart it in order to select it. To do this:

1. Right-click on **Compute** in the left navigation pane and select *Open in new tab*.

1. Find the triangle icon to the right of your compute cluster name and click it.
1. Wait a few minutes for the cluster to start.
1. Once the cluster is running, complete the steps above to select your cluster.

## Requirements

Please review the following requirements before starting the lesson:

- To run this notebook, you need to use one of the following Databricks runtime(s): `17.3.x-scala2.13`

## Classroom Setup

Before starting the demo, run the provided classroom setup script. This script will define configuration variables necessary for the demo.

In [0]:
%run ../Includes/Classroom-Setup-4

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


Loading batch 1 of 31...1 seconds


True

**Other Conventions:**

Throughout this demo, we'll refer to the object `DA`. This object, provided by Databricks Academy, contains variables such as your username, catalog name, schema name, working directory, and dataset locations. Run the code block below to view these details:

In [0]:
print(f"Username:          {DA.username}")
print(f"Catalog Name:      {DA.catalog_name}")
print(f"Schema Name:       {DA.schema_name}")
print(f"Working Directory: {DA.paths.working_dir}")
print(f"Dataset Location:  {DA.paths.datasets}")

Username:          labuser12730509_1763721946@vocareum.com
Catalog Name:      dbacademy
Schema Name:       labuser12730509_1763721946
Working Directory: /Volumes/dbacademy/ops/labuser12730509_1763721946@vocareum_com
Dataset Location:  NestedNamespace (wine_quality='/Volumes/dbacademy_wine_quality/v01', california_housing='/Volumes/dbacademy_california_housing/v01', airbnb='/Volumes/dbacademy_airbnb/v01', covid='/Volumes/dbacademy_covid/v01', retail='/Volumes/dbacademy_retail/v01', ecommerce='/Volumes/dbacademy_ecommerce/v01')


## Task 1: Create a Databricks Serverless Lakeflow Job in the UI

1. **Navigate to Jobs & Pipelines**:
   - In your Databricks workspace, click on the **Jobs & Pipelines** icon in the left sidebar.
   
2. **Create a New Job**:
   - Click on **Create** in the upper-right corner of the Jobs & Pipelines page and Select **Job**.
   <br/>

   **📌 Note:** Make sure to turn **off** the `Lakeflow Jobs UI` option at the top of the job so that the following instructions match the UI.
   <br/>

   - Name the job `Serverless lakeflow Jobs` or something similar for easy identification.

## Task 2: Add Tasks to the Job:

### Task 2.1: Add Tasks - 1 to the Lakeflow Jobs

1. **Create First Task**:
   - Name the task `01__Raw_Data_to_Bronze_DLT_Pipeline`.
   - Set **Type** to `Notebook`.
   - **Source** should be set to `Workspace`.
   - Set **Path** to the notebook for data quality assessment (e.g., `../04 - Data Orchestration and Querying Capabilities/Notebooks/01 - Raw Data to Bronze DLT Pipeline`).
   - Select an `Serverless` cluster for this task.
   - Click **Create Task**.
   - Notifications:
      - Add notification to send emails on failure (e.g., ` your-email@databricks.com`).

This task runs the Raw Data-to-Bronze pipeline to ingest data.

### Task 2.2: Set Conditional Check for Bronze Pipeline

1. **Create Conditional Task**:
   - Click on **Add Task**
   - Set **Type** to `If/else condition`.
   - Name the task `DLT1_condition`.
   - Condition: Set the expression to **&lcub;&lcub;tasks.01__Raw_Data_to_Bronze_DLT_Pipeline.values.DLT_SUCCESS_True&rcub;&rcub; == SUCCESS**
   - Set **Depends on** to `01__Raw_Data_to_Bronze_DLT_Pipeline` to ensure this task runs after data quality checks.
   - Click **Save Task**.

This task evaluates whether the Bronze pipeline execution succeeded or failed, determining the next steps.

### Task 2.3: Bronze to Silver DLT Pipeline

1. **Create Second Task**:
   - Click on **Add Task**
    - Set **Type** to `Notebook`.
   - Name the task `02__Bronze_to_Silver_DLT_Pipeline`.
   - **Source** should be set to `Workspace`.
   - Set **Path** to the notebook for data quality assessment (e.g., `../04 - Data Orchestration and Querying Capabilities/Notebooks/02 - Bronze to Silver DLT Pipeline`).
   - Select an `Serverless` cluster for this task.
   - Set **Depends** on to **`DLT1_condition(True)`**
   - Click **Create Task**.
   - Notifications:
      - Add notification to send emails on failure (e.g., ` your-email@databricks.com`).

This task processes data from Bronze to Silver.

### Task 2.4: Set Conditional Check for Silver Pipeline
1. **Create Conditional Task**:
   - Click on **Add Task**
   - Set **Type** to `If/else condition`.
   - Name the task `DLT2_condition`.
   - Condition: Set the expression to  **&lcub;&lcub;tasks.02__Bronze_to_Silver_DLT_Pipeline.values.DLT_SUCCESS_True&rcub;&rcub; == SUCCESS**
   - Set **Depends on** to `02__Bronze_to_Silver_DLT_Pipeline`.
   - Click **Save Task**.

This task evaluates whether the Silver pipeline execution succeeded or failed.

### Task 2.5: Silver to Gold DLT Pipeline

1. **Create Third Task**:
   - Click on **Add Task**
   - Set **Type** to `Notebook`.
   - Name the task `03__Silver_to_Gold_DLT_Pipeline`.
   - **Source** should be set to `Workspace`.
   - Set **Path** to the notebook for feature importance analysis (e.g., `../04 - Data Orchestration and Querying Capabilities/Notebooks/03 - Silver to Gold DLT Pipeline`).
   - Use the same cluster as the previous tasks.
   - Set **Depends on** to:
     - `DLT2_condition (True)`.
   - Notifications:
      - Add notification to send emails on failure (e.g., ` your-email@databricks.com`).
   - Click **Create Task**.

This task processes data from Silver to Gold.

### Task 2.6: Troubleshooting Notebook

1. **Create Fifth Task**:
   - Click on **Add Task**
   - Set **Type** to `Notebook`.
   - Name the task `troubleshooting`.
   - **Source** should be set to `Workspace`.
   - Set **Path** to the notebook for saving the final report (e.g., `../04 - Data Orchestration and Querying Capabilities/Notebooks/Troubleshooting`).
   - Use the same cluster as the previous tasks.
   - Set **Depends on** to both:
     - `DLT1_condition (False)` and `DLT2_condition (False)`.
   - Set **Run if dependencies** to "At least one succeeded" to ensure it saves the report regardless of the path taken.
   - Notifications:
      - Add notification to send emails on Success (e.g., ` your-email@databricks.com`).
   - Click **Create Task**.

This task runs a troubleshooting notebook to analyze and resolve pipeline issues. Example steps in the notebook include querying logs and providing remediation suggestions.

### Task 2.7: Enable Email Notifications

1. **Set up Notifications**:
   - In the job's configuration, navigate to the **Notifications** section.
   - Enable email notifications by adding your email to receive updates on job completion.

## Task 3: Trigger the Lakeflow Jobs Manually

1. **Run the Job**:
   - Go to the job in the Databricks UI and click on **Run Now** in the top-right corner to manually trigger the job. This will execute all tasks in the Lakeflow Job according to their dependencies and conditions.

## Task 4: Monitor the Lakeflow Jobs Execution

1. **Navigate to the Runs Tab**:
   - In the job interface, go to the **Runs** tab to view active and completed executions of the job.

2. **Observe Task Execution**:
   - Each task’s status is displayed in the **Runs** tab, where you can see which tasks are currently executing or have completed.
   - Click on each task to view its execution details and outputs, allowing you to troubleshoot and verify each stage.
   - Check the logs to see if the Lakeflow Jobs followed the correct path based on the unusual pattern detection condition.

## Task 5: Lineage: Viewing Data Lineage for a Table
Data lineage in Unity Catalog provides end-to-end visibility into how data is sourced, transformed, and consumed. With lineage information, you can:

- Understand the dependencies of your datasets.
- Identify the upstream and downstream impact of schema changes.
- Debug pipeline issues by tracing data flow through the system.
- Ensure compliance by auditing data usage and transformations.

**Benefits of Data Lineage**
- **Visibility:** Gain a comprehensive view of data flow across your pipeline.
- **Impact Analysis:** Determine how changes in one dataset affect downstream applications.
- **Governance and Compliance:** Track data transformations and usage for regulatory requirements.

#### Viewing Lineage in Unity Catalog
The following code helps you access the lineage information for a table directly in the Databricks UI.

In [0]:
# Generate the workspace URL dynamically
workspace_url = f"https://{spark.conf.get('spark.databricks.workspaceUrl')}"

# Define the table name for which to view the lineage
table_name = "order_table_gold"  # Replace with any other table name as needed

# Construct the URL for the data lineage page in Unity Catalog
lineage_url = f"{workspace_url}/explore/data/{DA.catalog_name}/{DA.schema_name}/{table_name}?activeTab=lineage"

# Print a user-friendly message with the lineage URL
print(f"Access the data lineage for the table '{table_name}' using the following URL:")

# Display the URL as a clickable link in Databricks
displayHTML(f'<a href="{lineage_url}" target="_blank">Click here to view the lineage for {table_name}</a>')

Access the data lineage for the table 'order_table_gold' using the following URL:


Click here to view the lineage for order_table_gold

## Conclusion

In this demo, you learned how to:
- Configure and execute a Databricks Lakeflow job with multiple tasks.
- Use dependencies and conditional paths to control the flow of tasks based on the conditions.
- Set up email notifications to stay updated on job execution.
- Trigger the Lakeflow job manually and monitor its execution.

This Lakeflow Jobs setup ensures robust automation for DLT pipelines with integrated troubleshooting and notification mechanisms. The conditional paths provide flexibility to handle success and failure scenarios efficiently, while monitoring and logging enhance visibility into pipeline executions.

Additionally, you explored how to leverage **data lineage** within Unity Catalog, enabling deeper insights into the relationships between datasets and transformations. This feature enhances governance, auditing, and troubleshooting across your Lakeflow Jobs.

&copy; 2025 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>